# Lyra 0.1 — обучение на Google Colab с сохранением в Google Drive

GPU берём у Colab, а **всё важное лежит в Google Drive** (`/content/drive/MyDrive/Lyra01/`): данные, токенайзер, чекпоинты, логи, отчёты, экспорт.
`/content` стирается при обрыве сессии — Drive нет. Если сессия отвалилась: **просто запусти ячейки заново (Runtime → Run all)**, и обучение продолжится с последнего сохранённого шага.

**Открыть в Colab:** https://colab.research.google.com/github/er1kosfunpay-sketch/Lyra0.1/blob/main/notebooks/train_colab.ipynb

Перед стартом: **Runtime → Change runtime type → GPU** (T4 / L4 / V100 / A100, нужно ≥12 ГиБ VRAM).

| Что | Где лежит | Размер |
|---|---|---|
| Данные `Den4ikAI/russian_dialogues_2` | `MyDrive/Lyra01/data/` | 1,14 ГиБ |
| Токенайзер (vocab 16384) | `MyDrive/Lyra01/tokenizer/` | < 1 МБ |
| Чекпоинты `latest/best/final.pt` | `MyDrive/Lyra01/checkpoints/ru_full_v1/` | 3 × 2,9 ГиБ |
| Замороженный конфиг | `MyDrive/Lyra01/configs/colab.json` | < 1 КБ |
| Логи / отчёты / экспорт | `MyDrive/Lyra01/{logs,reports,export}/` | ~ 0,5 ГиБ |
| Рабочая копия | `/content/Lyra01/` | стирается при обрыве |

**Объём:** модель 257 991 680 параметров, ctx 1024, vocab 16384. Полный прогон = **14 993 шага** (1 эпоха, 122,8 млн токенов). Сохранение каждые **500 шагов** → при обрыве теряется максимум 500 шагов.

**Три правила:**
1. Не удаляй папку `Lyra01` в Drive — там весь прогресс.
2. Не редактируй `configs/colab.json` в репозитории: настройки заморожены в Drive-копии, иначе resume-проверка `config_hash` откажет (это защита, а не баг).
3. Не меняй `--batch-size` / `--grad-accum` — их произведение должно оставаться 8, иначе меняется `TRAIN_STEPS`.


In [ ]:
from pathlib import Path
import os, json, shutil, subprocess, sys, time, math
from google.colab import drive

drive.mount('/content/drive')

REPO_URL = "https://github.com/er1kosfunpay-sketch/Lyra0.1.git"
RUN_NAME  = "ru_full_v1"
CONFIG    = "configs/colab.json"
DATASET_REPO = "Den4ikAI/russian_dialogues_2"

MICRO_BATCH, GRAD_ACCUM = 1, 8     # effective batch = 8  ->  TRAIN_STEPS = 14993
SAVE_EVERY, EVAL_EVERY  = 500, 1000
KEEP_LAST               = 1
DRIVE_BUDGET_GIB        = 15       # бесплатный Google Drive; при Google One подними

DRIVE = Path('/content/drive/MyDrive/Lyra01')
LOCAL = Path('/content/Lyra01')
REPO  = LOCAL / 'Lyra0.1'

for p in (DRIVE/'data', DRIVE/'tokenizer', DRIVE/'checkpoints'/RUN_NAME,
          DRIVE/'logs', DRIVE/'reports', DRIVE/'configs'):
    p.mkdir(parents=True, exist_ok=True)
LOCAL.mkdir(parents=True, exist_ok=True)

du = shutil.disk_usage(DRIVE)
print(json.dumps({
    'drive': str(DRIVE),
    'drive_total_gib': round(du.total / 1024**3, 2),
    'drive_free_gib':  round(du.free  / 1024**3, 2),
    'budget_gib': DRIVE_BUDGET_GIB,
}, indent=2))
if du.free > 0 and du.free < 13 * 1024**3:
    print('ВНИМАНИЕ: в Drive меньше 13 ГиБ свободно — чекпоинты могут не поместиться.')


## 1. Репозиторий и зависимости

`git pull` может изменить `configs/colab.json`, а чекпоинт хранит `config_hash` и сверяет его при resume.
Поэтому конфиг **замораживается в Drive** с первого запуска и всегда восстанавливается оттуда.


In [ ]:
if REPO.exists() and (REPO/'.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
elif REPO.exists() and any(REPO.iterdir()):
    raise RuntimeError(f'Refusing to clone over non-empty directory: {REPO}')
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
os.chdir(REPO)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[data,dev]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)

PERSIST_CONFIG = DRIVE / 'configs' / Path(CONFIG).name
if PERSIST_CONFIG.is_file():
    shutil.copy2(PERSIST_CONFIG, REPO / CONFIG)
    mode = 'frozen copy restored from Drive'
else:
    shutil.copy2(REPO / CONFIG, PERSIST_CONFIG)
    mode = 'current config frozen to Drive'
print('Config:', CONFIG, '->', mode)
print('HEAD:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())


## 2. GPU

Тренер сам выбирает bf16, где он поддерживается, иначе fp16 + GradScaler (T4 — fp16).
`gradient_checkpointing` в этом профиле выключен ради скорости: модель 258M при batch 1 × 1024 занимает ~7–8 ГиБ VRAM, на 15 ГиБ T4 запас есть. Если вдруг OOM **до первого чекпоинта** — поставь `true` в `MyDrive/Lyra01/configs/colab.json` и перезапусти ячейки.


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('CUDA не найден. Runtime → Change runtime type → GPU и перезапусти ноутбук (Runtime → Restart runtime).')

props = torch.cuda.get_device_properties(0)
vram = props.total_memory / 1024**3
if vram < 12:
    raise RuntimeError(f'{props.name}: {vram:.1f} GiB VRAM — профилю Lyra нужно минимум 12 ГиБ.')

cfg_json = json.loads((REPO / CONFIG).read_text(encoding='utf-8-sig'))
profile = {
    'gpu': props.name,
    'vram_gib': round(vram, 2),
    'free_vram_gib': round(torch.cuda.mem_get_info()[0] / 1024**3, 2),
    'bf16_supported': torch.cuda.is_bf16_supported(),
    'amp': 'bf16' if torch.cuda.is_bf16_supported() else 'fp16',
    'micro_batch': MICRO_BATCH,
    'grad_accum': GRAD_ACCUM,
    'effective_batch': MICRO_BATCH * GRAD_ACCUM,
    'context_length': cfg_json['context_length'],
    'gradient_checkpointing': cfg_json['gradient_checkpointing'],
    'parameter_count': 257991680,
}
print(json.dumps(profile, indent=2))
(DRIVE / 'reports' / 'gpu_profile.json').write_text(json.dumps(profile, indent=2), encoding='utf-8')


## 3. Данные — единственный источник `Den4ikAI/russian_dialogues_2`

Сначала восстанавливаем готовые сплиты из Drive (≈1–2 минуты), иначе собираем их заново через `scripts/prepare_data.py` и сохраняем в Drive.
Другие датасеты не поддерживаются: сборка падает с ошибкой, а не подмешивает fallback.


In [ ]:
LOCAL_DATA = LOCAL / 'data' / 'processed'
DRIVE_DATA = DRIVE / 'data'
REQUIRED = ('train.jsonl', 'validation.jsonl', 'test.jsonl', 'dataset_stats.json')
LOCAL_DATA.mkdir(parents=True, exist_ok=True)

def verify_single_source(stats):
    if stats.get('source_names') != ['ru_chat'] or stats.get('dataset', {}).get('repo') != DATASET_REPO:
        raise RuntimeError(f"Отказано: источник {stats.get('source_names')!r}, ожидался только {DATASET_REPO}.")
    if stats.get('fallback_used') or not stats.get('single_source'):
        raise RuntimeError('Отказано: сборка использовала fallback-источник.')
    if stats.get('build_state') != 'SUCCESS':
        raise RuntimeError(f"Состояние сборки: {stats.get('build_state')!r}, ожидался SUCCESS.")
    return stats

have_drive = all((DRIVE_DATA / n).is_file() for n in REQUIRED)
if have_drive:
    stats = verify_single_source(json.loads((DRIVE_DATA / 'dataset_stats.json').read_text(encoding='utf-8-sig')))
    for n in REQUIRED:
        src, dst = DRIVE_DATA / n, LOCAL_DATA / n
        if not dst.is_file() or dst.stat().st_size != src.stat().st_size:
            print(f'восстанавливаю {n} ({src.stat().st_size / 1024**3:.2f} ГиБ) ...', flush=True)
            shutil.copy2(src, dst)
    print('Данные восстановлены из Drive.')
else:
    print('В Drive готовых сплитов нет — собираю корпус с нуля (~10 мин, нужен Internet).')
    subprocess.run([sys.executable, 'scripts/prepare_data.py', '--out', str(LOCAL_DATA)], cwd=REPO, check=True)
    stats = verify_single_source(json.loads((LOCAL_DATA / 'dataset_stats.json').read_text(encoding='utf-8-sig')))
    for n in REQUIRED:
        shutil.copy2(LOCAL_DATA / n, DRIVE_DATA / n)
    print('Сплиты сохранены в Drive.')

shutil.copy2(LOCAL_DATA / 'dataset_stats.json', DRIVE / 'reports' / 'dataset_stats.json')
print(json.dumps({
    'dataset': DATASET_REPO,
    'other_datasets': 'NONE',
    'clean_dialogues': stats.get('clean_dialogues'),
    'removed_total': stats.get('removed_total'),
    'duplicates_removed': stats.get('duplicates_removed'),
    'splits': {k: v.get('conversations') for k, v in stats.get('splits', {}).items()},
    'train_gib': round((LOCAL_DATA / 'train.jsonl').stat().st_size / 1024**3, 2),
}, indent=2, ensure_ascii=False))


## 4. Токенайзер (vocab 16384)

Один и тот же файл `tokenizer.json` нужен и для обучения, и для resume: чекпоинт хранит его sha256-отпечаток и не примет другой токенайзер. Поэтому он лежит в Drive и восстанавливается байт-в-байт.


In [ ]:
LOCAL_TOK = REPO / 'artifacts' / 'tokenizer'
DRIVE_TOK = DRIVE / 'tokenizer'
LOCAL_TOK.mkdir(parents=True, exist_ok=True)

need_train = True
if (DRIVE_TOK / 'tokenizer.json').is_file() and (DRIVE_TOK / 'tokenizer_stats.json').is_file():
    shutil.copy2(DRIVE_TOK / 'tokenizer.json', LOCAL_TOK / 'tokenizer.json')
    shutil.copy2(DRIVE_TOK / 'tokenizer_stats.json', LOCAL_TOK / 'tokenizer_stats.json')
    need_train = False
    print('Токенайзер восстановлен из Drive.')

if need_train:
    subprocess.run([sys.executable, 'scripts/train_tokenizer.py',
                    '--data', str(LOCAL_DATA / 'train.jsonl'),
                    '--vocab-size', '16384', '--expect-vocab-size', '16384',
                    '--out', str(LOCAL_TOK / 'tokenizer.json'),
                    '--stats-out', str(LOCAL_TOK / 'tokenizer_stats.json'),
                    '--count-splits', 'auto'], cwd=REPO, check=True)
    shutil.copy2(LOCAL_TOK / 'tokenizer.json', DRIVE_TOK / 'tokenizer.json')
    shutil.copy2(LOCAL_TOK / 'tokenizer_stats.json', DRIVE_TOK / 'tokenizer_stats.json')
    print('Токенайзер обучен и сохранён в Drive.')

from lyra.tokenizer import LyraTokenizer, SPECIAL_TOKENS
tok = LyraTokenizer.from_file(LOCAL_TOK / 'tokenizer.json')
if tok.vocab_size != 16384:
    raise RuntimeError(f'vocab_size = {tok.vocab_size}, ожидалось 16384.')
for probe in ('Привет', 'Привет, как дела?', 'Здравствуйте! Рад тебя видеть.',
              'Что ты сейчас делаешь?', 'Мне сегодня очень грустно',
              'Давай немного поговорим', 'Я играю в Roblox'):
    ids = tok.encode(probe)
    back = tok.decode(ids)
    if back != probe or '<UNK>' in back:
        raise RuntimeError(f'Русский round-trip провален для {probe!r}: {back!r}')
report = json.loads((LOCAL_TOK / 'tokenizer_stats.json').read_text(encoding='utf-8-sig'))
if not report.get('verification', {}).get('ok', False):
    raise RuntimeError('Верификация токенайзера не пройдена.')
fingerprint = __import__('hashlib').sha256((LOCAL_TOK / 'tokenizer.json').read_bytes()).hexdigest()
print(json.dumps({'vocab_size': tok.vocab_size, 'special_tokens': list(SPECIAL_TOKENS),
                  'train_tokens': report['splits']['train']['tokens'],
                  'sha256': fingerprint}, indent=2, ensure_ascii=False))


## 5. Бюджет Google Drive

Бесплатный Google Drive = **15 ГиБ**. Расчёт: данные 1,14 + три чекпоинта по 2,9 + экспорт ≈ 10,4 ГиБ с запасом. Ячейка упадёт заранее, если места не хватает.


In [ ]:
from lyra.config import LyraConfig

cfg = LyraConfig.from_json(CONFIG)
params = cfg.parameter_count()
if params != 257991680:
    raise RuntimeError(f'Неожиданное число параметров: {params}')

def dir_bytes(p):
    total = 0
    p = Path(p)
    if p.exists():
        for f in p.rglob('*'):
            try:
                if f.is_file():
                    total += f.stat().st_size
            except OSError:
                pass
    return total

est_ckpt_gib = (params * 12 + 50_000_000) / 1024**3      # веса fp32 + AdamW (2 состояния)
plan = {
    'data_gib': dir_bytes(DRIVE / 'data') / 1024**3,
    'checkpoints_gib': est_ckpt_gib * 3,                  # latest + best + final
    'export_gib': (params * 2) / 1024**3,                 # экспорт в bfloat16
    'logs_reports_gib': (dir_bytes(DRIVE / 'logs') + dir_bytes(DRIVE / 'reports')) / 1024**3,
}
projected = sum(plan.values())
du = shutil.disk_usage(DRIVE)
free = min(du.free, DRIVE_BUDGET_GIB * 1024**3) if du.free > 0 else DRIVE_BUDGET_GIB * 1024**3
budget = {k: round(v, 2) for k, v in plan.items()}
budget.update({'projected_gib': round(projected, 2),
               'drive_free_gib': round(free / 1024**3, 2),
               'limit_gib': DRIVE_BUDGET_GIB,
               'est_ckpt_gib': round(est_ckpt_gib, 2)})
print(json.dumps(budget, indent=2))
(DRIVE / 'reports' / 'budget.json').write_text(json.dumps(budget, indent=2), encoding='utf-8')
if projected > free - 2 * 1024**3:
    raise RuntimeError('Бюджет Drive исчерпан: удали старые чекпоинты или освободи место.')
print('Drive budget OK')


## 6. Preflight

CUDA, точное число параметров, токенайзер, одиночный источник данных и быстрый forward/backward — всё до старта обучения.


In [ ]:
from lyra.config import LyraConfig
cfg = LyraConfig.from_json(CONFIG)
subprocess.run([sys.executable, 'scripts/colab_healthcheck.py', '--config', CONFIG,
                '--require-cuda', '--instantiate', '--sanity'], cwd=REPO, check=True)

tok_stats = json.loads((REPO / 'artifacts' / 'tokenizer' / 'tokenizer_stats.json').read_text(encoding='utf-8-sig'))
train_tokens = tok_stats['splits']['train']['tokens']
TRAIN_STEPS = math.ceil(train_tokens / (MICRO_BATCH * GRAD_ACCUM * cfg.context_length))
assert cfg.vocab_size == tok_stats['vocab_size'], (cfg.vocab_size, tok_stats['vocab_size'])
print(json.dumps({
    'train_tokens': train_tokens,
    'effective_batch': MICRO_BATCH * GRAD_ACCUM,
    'train_steps': TRAIN_STEPS,
    'parameter_count': cfg.parameter_count(),
    'save_every': SAVE_EVERY,
    'est_worst_case_loss_on_disconnect_steps': SAVE_EVERY,
}, indent=2))


## 7. Обучение

Чекпоинты пишутся **локально** (`/content`, быстро, атомарно с проверкой чтением), а сразу после каждого сохранения **синхронизируются в Google Drive** в фоне потока тренера — тренировка при этом не останавливается.

Порядок действий при обрыве сессии: `Runtime → Reconnect` (или новый GPU-runtime) → **Run all** сверху вниз. Ячейка данных, токенайзера и обучения восстановит всё из Drive и продолжит с `latest.pt`.


In [ ]:
CKPT_LOCAL = LOCAL / 'checkpoints' / RUN_NAME
CKPT_DRIVE = DRIVE / 'checkpoints' / RUN_NAME
CKPT_LOCAL.mkdir(parents=True, exist_ok=True)

def sync(src_dir, dst_dir, names=('latest.pt', 'best.pt', 'final.pt', 'training.jsonl')):
    # Атомарная запись чекпоинта в Drive: tmp -> os.replace. True = всё синхронизировано.
    src_dir, dst_dir = Path(src_dir), Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    for attempt in range(3):
        try:
            for n in names:
                src = src_dir / n
                if not src.is_file():
                    continue
                s = src.stat()
                dst = dst_dir / n
                if dst.is_file():
                    d = dst.stat()
                    if d.st_size == s.st_size and abs(d.st_mtime - s.st_mtime) < 2:
                        continue                      # уже синхронизировано
                tmp = dst.with_name(dst.name + '.syncing')
                shutil.copy2(src, tmp)
                os.replace(tmp, dst)
            (DRIVE / 'logs' / 'last_sync.json').write_text(json.dumps(
                {'ok': True, 'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
                 'source': str(src_dir)}, ensure_ascii=False), encoding='utf-8')
            return True
        except Exception as e:
            print(f'[sync] попытка {attempt + 1}/3: {e}', flush=True)
            time.sleep(10)
    (DRIVE / 'logs' / 'last_sync.json').write_text(json.dumps(
        {'ok': False, 'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
         'source': str(src_dir)}, ensure_ascii=False), encoding='utf-8')
    return False

def restore_from_drive(ckpt_local, ckpt_drive):
    ckpt_local, ckpt_drive = Path(ckpt_local), Path(ckpt_drive)
    if (ckpt_local / 'latest.pt').is_file():
        print('Найден локальный latest.pt (та же сессия):', ckpt_local / 'latest.pt')
        return 'local'
    if (ckpt_drive / 'latest.pt').is_file():
        for n in ('latest.pt', 'best.pt', 'training.jsonl'):
            src = ckpt_drive / n
            if src.is_file():
                print(f'Восстанавливаю {n} ({src.stat().st_size / 1024**3:.2f} ГиБ) из Drive ...', flush=True)
                shutil.copy2(src, ckpt_local / n)
        return 'drive'
    print('Чекпоинтов нет — будет запуск с нуля.')
    return 'fresh'

def run_training(cmd, ckpt_local, ckpt_drive, log_name):
    # Запуск тренера с построчным выводом, логом и синхронизацией каждого чекпоинта в Drive.
    log_local = LOCAL / 'logs' / log_name
    log_local.parent.mkdir(parents=True, exist_ok=True)
    started, all_synced = time.time(), True
    with log_local.open('a', encoding='utf-8') as lf:
        proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='')
            lf.write(line)
            lf.flush()
            try:
                rec = json.loads(line)
            except Exception:
                continue
            if 'checkpoint_saved' in rec:
                print(f"[sync] шаг {rec['checkpoint_saved']} -> Google Drive ...", flush=True)
                ok = sync(ckpt_local, ckpt_drive)
                all_synced = all_synced and ok
                if not ok:
                    print('[sync] !! НЕ СОХРАНИЛОСЬ В DRIVE — проверь квоту и сеть!', flush=True)
        code = proc.wait()
    all_synced = sync(ckpt_local, ckpt_drive) and all_synced
    try:
        shutil.copy2(log_local, DRIVE / 'logs' / log_name)
    except Exception as e:
        print('[sync] не удалось скопировать лог в Drive:', e)
    print(f'Код процесса: {code}; время {(time.time() - started) / 3600:.2f} ч; drive_sync_ok={all_synced}')
    if code:
        raise RuntimeError(f'Тренировка завершилась с кодом {code}; лог: {log_local}')
    return all_synced

restore_from_drive(CKPT_LOCAL, CKPT_DRIVE)

cmd = [sys.executable, 'scripts/train.py', '--config', CONFIG,
       '--tokenizer', str(REPO / 'artifacts' / 'tokenizer' / 'tokenizer.json'),
       '--data', str(LOCAL_DATA / 'train.jsonl'),
       '--validation', str(LOCAL_DATA / 'validation.jsonl'),
       '--stage', 'pretrain', '--steps', str(TRAIN_STEPS),
       '--batch-size', str(MICRO_BATCH), '--grad-accum', str(GRAD_ACCUM),
       '--lr', '3e-4', '--warmup-steps', '100',
       '--save-every', str(SAVE_EVERY), '--archive-every', '0',
       '--keep-last-checkpoints', str(KEEP_LAST), '--eval-every', str(EVAL_EVERY),
       '--out', str(CKPT_LOCAL)]
if (CKPT_LOCAL / 'latest.pt').is_file():
    cmd += ['--resume', str(CKPT_LOCAL / 'latest.pt')]
print('CMD:', ' '.join(cmd))
run_training(cmd, CKPT_LOCAL, CKPT_DRIVE, 'pretrain.log')


## 8. Прогресс — запускай в любой момент

Показывает шаги, скорость, ETA, историю validation loss и загрузку GPU. Скорость считается по истории запусков этой ячейки (нужно запустить дважды с паузой ≥2 минут).


In [ ]:
RUN_NAME = globals().get('RUN_NAME', 'ru_full_v1')
DRIVE = globals().get('DRIVE', Path('/content/drive/MyDrive/Lyra01'))
LOCAL = globals().get('LOCAL', Path('/content/Lyra01'))
CKPT_LOCAL = LOCAL / 'checkpoints' / RUN_NAME
CKPT_DRIVE = DRIVE / 'checkpoints' / RUN_NAME

log_path = CKPT_LOCAL / 'training.jsonl'
if not log_path.is_file():
    log_path = CKPT_DRIVE / 'training.jsonl'
if not log_path.is_file():
    print('Логов пока нет — обучение ещё не запускалось.')
else:
    recs = [json.loads(l) for l in log_path.read_text(encoding='utf-8').splitlines()
            if l.strip().startswith('{')]
    last = recs[-1]
    hist_path = DRIVE / 'logs' / 'progress_history.json'
    hist = json.loads(hist_path.read_text(encoding='utf-8')) if hist_path.is_file() else []
    hist.append({'t': time.time(), 'step': last['step']})
    hist = hist[-100:]
    hist_path.write_text(json.dumps(hist), encoding='utf-8')

    rate = None
    if len(hist) >= 2 and hist[-1]['t'] - hist[0]['t'] > 120:
        rate = (hist[-1]['step'] - hist[0]['step']) / (hist[-1]['t'] - hist[0]['t'])
    losses = [r['loss'] for r in recs if 'loss' in r]
    vals = [[r['step'], round(r['validation_loss'], 4)] for r in recs if 'validation_loss' in r]
    target = globals().get('TRAIN_STEPS', 14993)
    out = {
        'step': last['step'],
        'target': target,
        'percent': round(100 * last['step'] / target, 1),
        'tokens_seen': last.get('tokens_seen'),
        'loss_avg_last10': round(sum(losses[-10:]) / min(10, len(losses)), 4) if losses else None,
        'steps_per_sec': round(rate, 4) if rate else None,
        'eta_hours': round((target - last['step']) / rate / 3600, 2) if rate else None,
        'validation_history': vals[-6:],
        'gpu_util_recent': [r.get('gpu_utilization_percent') for r in recs
                            if 'gpu_utilization_percent' in r][-5:],
        'last_sync': json.loads((DRIVE / 'logs' / 'last_sync.json').read_text(encoding='utf-8'))
                     if (DRIVE / 'logs' / 'last_sync.json').is_file() else None,
        'local_ckpt_gib': round(dir_bytes(CKPT_LOCAL) / 1024**3, 2) if 'dir_bytes' in globals() else None,
        'drive_free_gib': round(shutil.disk_usage(DRIVE).free / 1024**3, 2),
    }
    print(json.dumps(out, indent=2, ensure_ascii=False))


## 9. Оценка и экспорт

`evaluate_russian.py` выдаёт PASS/FAIL по русскому языку, `<UNK>`, копированию промпта и repetition-loop'ам. Экспорт — веса в bfloat16 + конфиг + токенайзер, всё улетает в Drive.


In [ ]:
ckpt_eval = CKPT_LOCAL / 'final.pt' if (CKPT_LOCAL / 'final.pt').is_file() else CKPT_LOCAL / 'latest.pt'
if not ckpt_eval.is_file():
    raise FileNotFoundError('Чекпоинта нет — обучение ещё не запускалось.')

eval_out = REPO / 'logs' / 'russian_eval.json'
eval_out.parent.mkdir(parents=True, exist_ok=True)
proc = subprocess.run([sys.executable, 'scripts/evaluate_russian.py',
                       '--checkpoint', str(ckpt_eval),
                       '--tokenizer', str(REPO / 'artifacts' / 'tokenizer' / 'tokenizer.json'),
                       '--out', str(eval_out)], cwd=REPO)
print('evaluate_russian returncode:', proc.returncode)
if eval_out.is_file():
    shutil.copy2(eval_out, DRIVE / 'reports' / 'russian_eval.json')
    ev = json.loads(eval_out.read_text(encoding='utf-8'))
    print(json.dumps({'overall': ev.get('overall'), 'failed_checks': ev.get('failed_checks'),
                      'mean_cyrillic_share': ev.get('mean_cyrillic_share'),
                      'total_unk_ids': ev.get('total_unk_ids'),
                      'special_token_leaks': ev.get('special_token_leaks'),
                      'repetition_loops': ev.get('repetition_loops')}, indent=2, ensure_ascii=False))

export_local = LOCAL / 'lyra_export'
subprocess.run([sys.executable, 'scripts/export_model.py', '--config', CONFIG,
                '--checkpoint', str(ckpt_eval),
                '--tokenizer', str(REPO / 'artifacts' / 'tokenizer' / 'tokenizer.json'),
                '--out', str(export_local), '--dtype', 'bfloat16'], cwd=REPO, check=True)
export_drive = DRIVE / 'export' / RUN_NAME
if export_drive.exists():
    shutil.rmtree(export_drive)
shutil.copytree(export_local, export_drive)
print('Экспорт в Drive:', [f'{p.name} {p.stat().st_size / 1024**2:.0f} MiB'
                           for p in sorted(export_drive.iterdir())])


## 10. Если сессия оборвалась

1. `Runtime → Disconnect and delete runtime` (или просто новый runtime) — ничего не теряется: данные, токенайзер, чекпоинты и логи в Drive.
2. Выбери GPU (`Runtime → Change runtime type → GPU`) и нажми **Runtime → Run all**.
3. Ячейка обучения сама найдёт `latest.pt` (сначала локально, потом в Drive) и продолжит с того же шага. Потери ≤ 500 шагов.
4. Проверь строку `drive_sync_ok=True` в конце ячейки обучения и `last_sync` в ячейке прогресса.

**Квота.** Полный прогон занимает ~25–35 часов GPU (T4 ≈ 2–4 с/шаг при выключенном gradient checkpointing). Бесплатный Colab даёт сессию до ~12 часов GPU, поэтому хватит 2–4 сессий. Недельная квота бесплатного аккаунта на это не рассчитана — смотри остаток в `Runtime → Manage sessions` / на странице Quota.

**Если квота тает.** Цель `--steps` не входит в отпечаток конфига, поэтому её можно уменьшать прямо в ячейке обучения и между сессиями (7500 шагов ≈ половина эпохи) — resume докрутит до новой цели. Сам `configs/colab.json` между сессиями не меняй.


## 11. Опционально: второй этап SFT (маска только на ответах ассистента)

Запускай **только после** завершения pretrain. Тот же корпус, та же конфигурация, но loss считается лишь на токенах `<ASSISTANT>` — модель перестаёт «дописывать реплику пользователя» и лучше следует инструкциям. Прервался — запусти ячейку снова, она продолжит свой `latest.pt`.


In [ ]:
SFT_RUN = 'ru_sft_v1'
CKPT_LOCAL_S = LOCAL / 'checkpoints' / SFT_RUN
CKPT_DRIVE_S = DRIVE / 'checkpoints' / SFT_RUN
CKPT_LOCAL_S.mkdir(parents=True, exist_ok=True)

if restore_from_drive(CKPT_LOCAL_S, CKPT_DRIVE_S) == 'fresh':
    if not (CKPT_LOCAL / 'final.pt').is_file():
        raise RuntimeError('Нет final.pt от pretrain — сначала закончи основной прогон.')
    print('SFT стартует с final.pt pretrain (--reset-stage).')

sft_cmd = [sys.executable, 'scripts/train.py', '--config', CONFIG,
           '--tokenizer', str(REPO / 'artifacts' / 'tokenizer' / 'tokenizer.json'),
           '--data', str(LOCAL_DATA / 'train.jsonl'),
           '--validation', str(LOCAL_DATA / 'validation.jsonl'),
           '--stage', 'sft', '--steps', '4000',
           '--batch-size', str(MICRO_BATCH), '--grad-accum', str(GRAD_ACCUM),
           '--lr', '1e-4', '--warmup-steps', '50',
           '--save-every', str(SAVE_EVERY), '--archive-every', '0',
           '--keep-last-checkpoints', str(KEEP_LAST), '--eval-every', '500',
           '--out', str(CKPT_LOCAL_S)]
if (CKPT_LOCAL_S / 'latest.pt').is_file():
    sft_cmd += ['--resume', str(CKPT_LOCAL_S / 'latest.pt')]
elif (CKPT_LOCAL / 'final.pt').is_file():
    sft_cmd += ['--resume', str(CKPT_LOCAL / 'final.pt'), '--reset-stage']
print('CMD:', ' '.join(sft_cmd))
run_training(sft_cmd, CKPT_LOCAL_S, CKPT_DRIVE_S, 'sft.log')
